# General Tool-Using Agent With Groq And Gradio

This notebook builds a general agent.

It is not only a restaurant agent.

It can answer normal questions using the LLM, and it can use tools when the model needs current information.

Tools included:

- current date and time tool
- weather tool using Open-Meteo, free and no API key
- web search tool using DuckDuckGo HTML search, free and no API key
- Gradio app

## Easy Idea

An LLM is smart, but it may not know live information.

For example, it may not know:

- today's date
- current weather
- today's news
- latest updates after its training date

So we give the agent tools.

Daily life example: a teacher may know many things, but for today's weather the teacher checks a weather app. The weather app is a tool.

## Free Tool Suggestions

| Need | Free-friendly tool |
| --- | --- |
| Current date/time | Python `datetime` |
| Weather | Open-Meteo API, no key |
| Web search | DuckDuckGo search, no key |
| LLM | Groq API key |

Important: free tools can still have limits or temporary blocking if used too much.

## Step 1: Imports

We import Python tools, LangChain tools, LangGraph, Groq, and Gradio.

In [ ]:
import json
import os
import re
import urllib.parse
import urllib.request
from datetime import datetime
from html import unescape
from zoneinfo import ZoneInfo

import gradio as gr
from dotenv import load_dotenv
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langgraph.graph import START, StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

## Step 2: Load Groq Key

Add your Groq key in `.env`:

```env
GROQ_API_KEY=your_key_here
```

Do not upload `.env` to GitHub.

In [ ]:
load_dotenv()

# Disable LangSmith tracing for this learning notebook.
# If tracing is on without a valid LANGCHAIN_API_KEY, LangChain shows 401 Unauthorized warnings.
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_TRACING"] = "false"

print("GROQ_API_KEY:", "set" if os.getenv("GROQ_API_KEY") else "missing")

## Step 3: Create Tools

A tool is a Python function that the LLM can call.

The LLM does not directly know live weather or live search results.

When needed, it calls these tools and then uses their results to answer.

In [ ]:
duckduckgo_search = DuckDuckGoSearchRun()


@tool
def get_current_datetime(timezone: str = "Asia/Karachi") -> str:
    """Get the current date and time for a timezone. Use for questions about today, date, time, or now."""
    try:
        now = datetime.now(ZoneInfo(timezone))
    except Exception:
        now = datetime.now()

    return now.strftime("%A, %B %d, %Y %I:%M %p")


@tool
def get_weather(city: str) -> str:
    """Get current weather for a city using Open-Meteo. Use for weather questions."""
    city = city.strip()
    if not city:
        return "Please provide a city name."

    geocode_url = "https://geocoding-api.open-meteo.com/v1/search?" + urllib.parse.urlencode({
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    })

    with urllib.request.urlopen(geocode_url, timeout=15) as response:
        geocode_data = json.loads(response.read().decode("utf-8"))

    results = geocode_data.get("results", [])
    if not results:
        return f"I could not find weather coordinates for {city}."

    place = results[0]
    latitude = place["latitude"]
    longitude = place["longitude"]
    place_name = place.get("name", city)
    country = place.get("country", "")

    weather_url = "https://api.open-meteo.com/v1/forecast?" + urllib.parse.urlencode({
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,relative_humidity_2m,wind_speed_10m",
        "timezone": "auto"
    })

    with urllib.request.urlopen(weather_url, timeout=15) as response:
        weather_data = json.loads(response.read().decode("utf-8"))

    current = weather_data.get("current", {})
    temperature = current.get("temperature_2m")
    humidity = current.get("relative_humidity_2m")
    wind = current.get("wind_speed_10m")

    return f"Current weather in {place_name}, {country}: temperature {temperature} C, humidity {humidity}%, wind speed {wind} km/h."


@tool
def web_search(query: str) -> str:
    """Search the web for current or latest information. Use for latest news, recent events, or facts that may have changed."""
    query = query.strip()
    if not query:
        return "Please provide a search query."

    try:
        return duckduckgo_search.invoke(query)
    except Exception as error:
        return f"Search tool error: {error}. Try again or use a different query."


## Step 4: Test Tools Directly

Before connecting tools to the LLM, test them like normal Python functions.

In [ ]:
print(get_current_datetime.invoke({"timezone": "Asia/Karachi"}))

In [ ]:
print(get_weather.invoke({"city": "Karachi"}))

In [ ]:
print(web_search.invoke({"query": "latest AI news today"}))

## Step 5: Create LLM And Bind Tools

`bind_tools` means we give the LLM permission to call these tools.

The LLM will decide when to call a tool.

Example:

- If user asks today's date, call date tool.
- If user asks weather, call weather tool.
- If user asks latest news, call search tool.
- If user asks a normal concept question, answer directly.

In [ ]:
tools = [get_current_datetime, get_weather, web_search]

# This model is more reliable for tool calling than smaller instant models.
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
llm_with_tools = llm.bind_tools(tools)

## Step 6: Build LangGraph Tool Agent

This graph has two main parts:

1. `assistant`: the LLM decides what to do
2. `tools`: LangGraph runs the selected tool

If the LLM calls a tool, the graph goes to the tool node and then back to the assistant.

If no tool is needed, the graph ends.

In [ ]:
system_prompt = """
You are a helpful general assistant.
Use tools when the user asks for current date, current time, weather, latest news, recent events, or live/current information.
If the answer does not need current information, answer directly.
When using web search, summarize the result and mention that it came from search results.
Keep answers simple and beginner friendly.
"""


def assistant_node(state: MessagesState):
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


tool_node = ToolNode(tools)

tool_agent_builder = StateGraph(MessagesState)
tool_agent_builder.add_node("assistant", assistant_node)
tool_agent_builder.add_node("tools", tool_node)

tool_agent_builder.add_edge(START, "assistant")
tool_agent_builder.add_conditional_edges("assistant", tools_condition)
tool_agent_builder.add_edge("tools", "assistant")

tool_agent = tool_agent_builder.compile()

## Step 7: Test The Agent

Try questions that need tools and questions that do not.

Important: sometimes an LLM can fail to format a tool call correctly. The helper function below uses simple safe routing first for date, weather, and search questions. This keeps the notebook working while you are learning.

In [ ]:
def safe_general_agent_answer(user_message: str) -> str:
    text = user_message.lower()

    if "weather" in text:
        city = "Karachi"
        match = re.search(r"weather (?:in|of|for) ([a-zA-Z ]+)", user_message, re.IGNORECASE)
        if match:
            city = match.group(1).strip()
        return get_weather.invoke({"city": city})

    if "date" in text or "time" in text or "today" in text:
        return get_current_datetime.invoke({"timezone": "Asia/Karachi"})

    if "latest" in text or "news" in text or "search" in text or "current" in text:
        return web_search.invoke({"query": user_message})

    response = tool_agent.invoke({"messages": [HumanMessage(content=user_message)]})
    return response["messages"][-1].content

In [ ]:
safe_general_agent_answer("What is today's date in Pakistan?")

In [ ]:
safe_general_agent_answer("What is the current weather in Karachi?")

In [ ]:
safe_general_agent_answer("Search latest AI news today and summarize it.")

## Step 8: Gradio App

Now we make a simple app.

Type any question.

The agent will decide if it should answer directly or use a tool.

In [ ]:
def general_agent_app(user_message: str) -> str:
    if not user_message.strip():
        return "Please type a question."

    try:
        return safe_general_agent_answer(user_message)
    except Exception as error:
        return f"Error: {error}"

In [ ]:
demo = gr.Interface(
    fn=general_agent_app,
    inputs=gr.Textbox(label="Ask anything", placeholder="Example: What is the weather in Karachi today?"),
    outputs=gr.Textbox(label="Agent Answer"),
    title="General Tool-Using Agent",
    description="Groq + LangGraph + tools for date, weather, and search."
)

demo.launch()

## What You Built

You built a general agent.

The LLM is the brain.

The tools are helpers.

LangGraph is the manager that moves between the brain and the tools.

Gradio is the user interface.

This is the basic pattern for many real agent apps.